# Step 4 Sanity Checks — Training EDA

This notebook verifies the integrity of the training pipeline after all 29 event models have been trained.

**Date generated:** 2026-04-22

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import joblib

from models import build_training_matrix, split_by_season

matches   = pd.read_parquet('../data/processed/matches.parquet')
features  = pd.read_parquet('../data/processed/features.parquet')
labels    = pd.read_parquet('../data/processed/labels.parquet')
catalog   = pd.read_parquet('../data/processed/events_catalog.parquet')
metrics   = pd.read_parquet('../data/processed/metrics.parquet')

features_with_teams = features.merge(
    matches[['match_id', 'team1', 'team2']], on='match_id', how='left'
)
splits = split_by_season(matches)

print('Data loaded OK')
print(f"Matches: {len(matches)} | Features: {features.shape[1]} cols | Labels: {len(labels)} rows")

## 1. Swap unit test

For team-scope events, `build_training_matrix` must swap `t1_` ↔ `t2_` columns when the team-under-prediction is `team2`. After the swap, `t1_form_win_rate` (and `h2h_t1_win_rate`) must always reflect the target team, regardless of whether they were originally `team1` or `team2`.

In [ ]:
fake_features = pd.DataFrame({
    'match_id': ['m1'],
    'team1': ['A'],
    'team2': ['B'],
    't1_form_win_rate': [0.8],
    't2_form_win_rate': [0.2],
    'h2h_t1_win_rate': [0.6],
    'h2h_t2_win_rate': [0.4],
    'venue_avg_first_innings_score': [150.0],
})

fake_labels = pd.DataFrame({
    'match_id': ['m1', 'm1'],
    'event_id': ['team_wins_match', 'team_wins_match'],
    'team': ['A', 'B'],
    'outcome': [1, 0],
})

fake_catalog = pd.DataFrame({
    'event_id': ['team_wins_match'], 'scope': ['team']
})

X, y, mids = build_training_matrix(
    'team_wins_match', fake_catalog, fake_labels, fake_features, ['m1']
)

# Team A (team1) → no swap → t1_form_win_rate stays 0.8
assert X.iloc[0]['t1_form_win_rate'] == 0.8, "Row 0 t1_form_win_rate mismatch"

# Team B (team2) → swap → t1_form_win_rate becomes 0.2
assert X.iloc[1]['t1_form_win_rate'] == 0.2, "Row 1 t1_form_win_rate mismatch"

# Matchup features also swap: h2h_t1_win_rate for B becomes 0.4
assert X.iloc[1]['h2h_t1_win_rate'] == 0.4, "Row 1 h2h_t1_win_rate mismatch"

print('Swap unit test PASSED')

## 2. Split counts

Time-aware split sizes. Total must equal the full matches table.

In [ ]:
for name, ids in splits.items():
    print(f"{name:>5}: {len(ids)} matches")

total = sum(len(v) for v in splits.values())
assert total == len(matches), f"Split total {total} != matches {len(matches)}"
print(f"\nTotal: {total} — OK")

## 3. Label availability per split

Pick a handful of diverse events and confirm that labels exist in every split with reasonable positive rates.

In [ ]:
sample_events = [
    'team_wins_match', 'match_sixes_gte_15', 'any_fifty',
    'any_century', 'match_goes_super_over'
]

for ev in sample_events:
    for split_name, split_ids in splits.items():
        sub = labels[
            (labels['event_id'] == ev) & (labels['match_id'].isin(split_ids))
        ]['outcome'].dropna()
        pos_rate = sub.mean() if len(sub) > 0 else float('nan')
        print(f"{ev:25s} {split_name:>5}: n={len(sub)}  pos_rate={pos_rate:.3f}")

## 4. Models beating baseline

Count how many calibrated models achieve lower log-loss than the constant base-rate baseline on the **test** set. Print the laggards for inspection.

In [ ]:
beats = metrics['test_beats_baseline'].sum()
total = len(metrics)
print(f"Beating baseline: {beats} / {total}")

laggards = metrics[~metrics['test_beats_baseline']][
    ['event_id', 'test_logloss', 'test_logloss_baseline', 'status']
]
print("\nEvents NOT beating baseline:")
print(laggards.to_string(index=False))

## 5. Calibration spot-check

For the best-improvement event (lowest `test_logloss` vs baseline), bin predicted probabilities into deciles and compare mean predicted probability to empirical frequency within each bin.

In [ ]:
best_event = metrics.loc[metrics['test_logloss'].idxmin(), 'event_id']
print(f"Best-improvement event: {best_event}")

from models import build_training_matrix, evaluate
from sklearn.metrics import brier_score_loss

model = joblib.load(f'../models/{best_event}.joblib')
X_test, y_test, _ = build_training_matrix(
    best_event, catalog, labels, features_with_teams, splits['test']
)

if len(X_test) == 0:
    print('No test data for this event.')
else:
    probs = model.predict_proba(X_test)[:, 1]
    df = pd.DataFrame({'prob': probs, 'actual': y_test})
    df['bin'] = pd.qcut(df['prob'], q=10, duplicates='drop')
    calib = df.groupby('bin').agg(mean_pred=('prob', 'mean'),
                                  mean_actual=('actual', 'mean'),
                                  n=('actual', 'count')).reset_index()
    print(calib.to_string(index=False))

## 6. No-leakage re-verification

Confirm that the model uses only pre-match information by asserting that predictions on *future* matches do not depend on data from those matches.

Proxy test: train a model on the first half of the training set and predict on the second half; the predictions must be identical to training on the full training set (because both use only pre-match history).  We verify by loading an already-trained model and confirming its features are built from `features.parquet` which itself was generated with `cutoff_date < match_date`.

In [ ]:
import random

random.seed(42)
check_events = random.sample(catalog['event_id'].tolist(), 3)

for ev in check_events:
    scope = catalog[catalog['event_id'] == ev]['scope'].iloc[0]
    # Build matrix on a random test match
    test_mids = splits['test']
    mid = random.choice(test_mids)
    X_ev, y_ev, _ = build_training_matrix(
        ev, catalog, labels, features_with_teams, [mid]
    )
    assert len(X_ev) > 0, f"{ev}: no features for {mid}"
    # Every column in X_ev must also exist in features.parquet
    assert set(X_ev.columns).issubset(set(features.columns)), f"{ev}: extra columns found"
    print(f"{ev}: leakage check passed")

print("\nAll leakage checks PASSED")

## 7. Feature-importance spot-check (top 5 models)

Dig into the underlying XGBoost estimator of each calibrated pipeline and print the top-10 most important features. A healthy model should show a diverse mix (venue, team form, player stats, head-to-head) rather than one or two features dominating everything.

In [ ]:
metrics['improvement'] = metrics['test_logloss_baseline'] - metrics['test_logloss']
top5 = metrics.sort_values('improvement', ascending=False).head(5)

print('Top 5 models by test improvement over baseline')
print('=' * 60)

for _, row in top5.iterrows():
    ev = row['event_id']
    print(f"\n>>> {ev}  (improvement: {row['improvement']:.4f})")

    model = joblib.load(f'../models/{ev}.joblib')

    if not hasattr(model, 'named_steps'):
        print('    Flat predictor — no feature importances.')
        continue

    clf = model.named_steps['xgb']

    if hasattr(clf, 'calibrated_classifiers_') and len(clf.calibrated_classifiers_) > 0:
        base_est = clf.calibrated_classifiers_[0].estimator
    else:
        print('    No calibrated classifiers found.')
        continue

    X_train, _, _ = build_training_matrix(
        ev, catalog, labels, features_with_teams, splits['train']
    )
    if X_train.empty:
        print('    No training data.')
        continue

    feature_names = X_train.columns.tolist()
    importances = base_est.feature_importances_

    fi = pd.DataFrame({
        'feature': feature_names,
        'importance': importances,
    }).sort_values('importance', ascending=False)

    print(fi.head(10).to_string(index=False))